# Workshop: Data Reading & File Formats

Practice reading CSV, JSON, and Parquet files from Unity Catalog Volumes, using `read_files()` for SQL-native access, and creating Bronze Delta tables with CTAS.

**Prerequisite:** Complete the `02 — Data Reading & File Formats` demo before starting this lab.

| Task | Topic |
|------|-------|
| 1 | CSV — inferSchema |
| 2 | CSV — manual StructType schema |
| 3 | JSON — partial manual schema |
| 4 | Parquet — self-describing format |
| 5 | Performance — inferSchema vs manual |
| 6 | read_files() — SQL-native reader |
| 7 | CTAS — create Bronze table from SELECT |
| 8 | CTAS with data transformations |

## Setup

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

In [0]:
CUSTOMERS_CSV    = f"{DATASET_PATH}/customers/customers.csv"
ORDERS_JSON      = f"{DATASET_PATH}/orders/orders_batch.json"
PRODUCTS_PARQUET = f"{DATASET_PATH}/products/products.parquet"

display(spark.createDataFrame([
    ("CATALOG",           CATALOG),
    ("BRONZE_SCHEMA",     BRONZE_SCHEMA),
    ("CUSTOMERS_CSV",     CUSTOMERS_CSV),
    ("ORDERS_JSON",       ORDERS_JSON),
    ("PRODUCTS_PARQUET",  PRODUCTS_PARQUET),
], ["Variable", "Value"]))

## Task 1: Read CSV with inferSchema

Read `customers.csv` into a Spark DataFrame using automatic schema inference.

**What you need to do:** Use `spark.read.format("csv")` with `header=true` and `inferSchema=true`. Store the result in `customers_auto_df`.

**Guidance — Task 01**

The goal is to load a CSV file using Spark's built-in schema inference.

**Format reader pattern**
`spark.read` is the entry point for batch reads. You chain `.format("csv")` to select the format, then add `.option()` calls for reader configuration, and finally `.load(path)` to trigger the read. The `header=true` option tells Spark to treat the first row as column names rather than data.

**How inferSchema works**
When `inferSchema=true`, Spark makes two passes over the file: the first pass samples values to detect column types, the second pass reads the data with the inferred schema. On small files this is fast, but on large cloud files it doubles the I/O cost and read time.

**Things to think about**
- What types did Spark infer for each column? Are they correct?
- What would happen if the file had no header row and `header=false`?
- Would Spark correctly infer `TimestampType` from a date string like `2023-01-15`?

In [0]:
customers_auto_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CUSTOMERS_CSV)
)

customers_auto_df.printSchema()
display(customers_auto_df.limit(5))

In [0]:
# -- Validation --
assert customers_auto_df.count() > 0, "DataFrame should not be empty"
assert "customer_id" in customers_auto_df.columns, "Expected column 'customer_id'"
print(f"Task 1 OK: {customers_auto_df.count()} customers loaded. Inferred schema: {customers_auto_df.dtypes}")

## Task 2: Read CSV with Manual StructType Schema

Re-read the same CSV file using an explicit `StructType` schema. This removes the double-scan penalty of `inferSchema`.

**What you need to do:**
1. Define `customers_schema` using `StructType` and `StructField`
2. Load the CSV using `.schema(customers_schema)` instead of `inferSchema=true`

**Guidance — Task 02**

The goal is to define an explicit schema and pass it to the CSV reader.

**StructType syntax**
A schema is built as a list of `StructField(name, type, nullable)` objects inside a `StructType`. Types are imported from `pyspark.sql.types` — already done in the configuration cell above.

```python
StructType([
    StructField("column_name", StringType(), nullable=True),
    StructField("amount",      DoubleType(), nullable=True),
])
```

**Columns for customers**
`customer_id` (String, nullable=False), `first_name` (String), `last_name` (String), `city` (String), `email` (String), `country` (String), `registration_date` (TimestampType)

**Schema vs inferSchema**
When you pass `.schema(customers_schema)`, Spark reads the file in a single pass — the schema is already known. Type coercion happens at read time: if `registration_date` is a string in the file, Spark parses it as Timestamp automatically.

**Things to think about**
- What happens if you set `nullable=False` for a column that actually has null values?
- Compare the data types between `customers_auto_df` (Task 1) and `customers_df` (Task 2) — are they identical?

In [0]:
customers_schema = StructType([
    StructField("customer_id",        StringType(),    False),
    StructField("first_name",         StringType(),    True),
    StructField("last_name",          StringType(),    True),
    StructField("city",               StringType(),    True),
    StructField("email",              StringType(),    True),
    StructField("country",            StringType(),    True),
    StructField("registration_date",  TimestampType(), True),
])

In [0]:
customers_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)

customers_df.printSchema()
display(customers_df.limit(5))

In [0]:
# -- Validation --
reg_date_type = dict(customers_df.dtypes).get("registration_date")
assert reg_date_type == "timestamp", f"registration_date should be TimestampType, got {reg_date_type}"
assert customers_df.count() > 0
print(f"Task 2 OK: {customers_df.count()} rows loaded. registration_date type = {reg_date_type}")

## Task 3: Read JSON with Partial Manual Schema

Read `orders_batch.json` using a partial schema — select only the columns you need instead of loading every field.

**What you need to do:**
1. Define `orders_schema` with 5 columns: `order_id`, `customer_id`, `order_datetime`, `total_amount`, `payment_method`
2. Load the JSON file using this partial schema

**Guidance — Task 03**

The goal is to read a JSON file while loading only a subset of the available fields.

**Partial schema with JSON**
Unlike CSV, JSON files are self-describing — every record contains all field names. When you pass a `.schema()` to the JSON reader, Spark reads only the fields listed in the schema and silently ignores the rest. This reduces memory usage and speeds up reads on wide JSON records with many unused fields.

**Type choices at the Bronze layer**
It is common to keep timestamps as `StringType()` in Bronze — e.g., `order_datetime` — to avoid parse errors on malformed records. Type conversions happen in Silver where you have more control over error handling.

**Columns to include**
`order_id` (String), `customer_id` (String), `order_datetime` (String), `total_amount` (Double), `payment_method` (String)

**Things to think about**
- What happens to the other JSON fields not in the schema (e.g., `product_id`, `quantity`)?
- Why might you keep `order_datetime` as StringType in Bronze instead of TimestampType?

In [0]:
orders_schema = StructType([
    StructField("order_id",        StringType(), True),
    StructField("customer_id",     StringType(), True),
    StructField("order_datetime",  StringType(), True),
    StructField("total_amount",    DoubleType(), True),
    StructField("payment_method",  StringType(), True),
])

In [0]:
orders_df = (
    spark.read
    .format("json")
    .schema(orders_schema)
    .load(ORDERS_JSON)
)

orders_df.printSchema()
display(orders_df.limit(5))

In [0]:
# -- Validation --
assert orders_df.count() > 0
assert "total_amount" in orders_df.columns
assert len(orders_df.columns) == 5, f"Expected 5 columns, got {len(orders_df.columns)}: {orders_df.columns}"
print(f"Task 3 OK: {orders_df.count()} orders loaded with {len(orders_df.columns)} selected columns")

## Task 4: Read Parquet

Read `products.parquet`. Unlike CSV and JSON, Parquet files embed the schema in the file footer — no `inferSchema` or manual schema definition is needed.

**What you need to do:** Read the file using `spark.read.format("parquet")` and inspect the schema.

**Guidance — Task 04**

The goal is to read a self-describing binary columnar format without any schema configuration.

**Why Parquet is different**
Parquet is a columnar binary format that stores full type information in its file footer. When Spark opens a Parquet file, it reads the schema from metadata before reading any row data — there is no sampling, no double-scan, and no risk of type misdetection. You get the exact types that were written by the original producer.

**No schema options needed**
The reader call is simpler than CSV or JSON: `.format("parquet").load(path)`. Spark handles schema discovery automatically.

**Parquet in Databricks**
Delta Lake stores all data as Parquet files under the hood. When you read a Delta table with `spark.table()`, you are ultimately reading Parquet files with Delta transaction log metadata on top.

**Things to think about**
- How many columns does `products_df` have compared to `orders_df` from Task 3?
- Why is Parquet the default storage format in Delta Lake?
- What does "columnar storage" mean for analytical query performance?

In [0]:
products_df = (
    spark.read
    .format("parquet")
    .load(PRODUCTS_PARQUET)
)

products_df.printSchema()
display(products_df.limit(5))

In [0]:
# -- Validation --
assert products_df.count() > 0
assert "product_id" in products_df.columns
print(f"Task 4 OK: {products_df.count()} products. Schema from Parquet footer: {products_df.dtypes}")

## Task 5: Performance — inferSchema vs Manual Schema

Measure the wall-clock read time for both approaches on the same CSV file and print a side-by-side comparison.

**What you need to do:**
1. Time a CSV read with `inferSchema=true` using `time.time()` around a `.count()` action
2. Time the same read with `customers_schema` (from Task 2)
3. Print results — a comparison table is provided for you

**Guidance — Task 05**

The goal is to demonstrate the cost difference between schema inference and an explicit schema on a concrete timing measurement.

**Timing pattern**
Spark transformations are lazy — nothing runs until you call an **action** such as `.count()`, `.collect()`, or `.show()`. Always wrap the timing around an action:

```python
start     = time.time()
df        = spark.read...     # transformation (lazy — no work yet)
row_count = df.count()        # action — forces execution
elapsed   = time.time() - start
```

**Interpreting the results**
On a small training dataset the absolute difference may be small due to JVM warmup and local filesystem caching. The **pattern** is what matters: `inferSchema` always performs two filesystem scans, while a manual schema performs one. On 100 GB files in cloud object storage, the difference is measured in minutes and dollars.

**Things to think about**
- On a real 100 GB CSV with 200 columns in ADLS, how would the cost difference change?
- Is there ever a case where `inferSchema=true` is the right choice in production?

In [0]:
start_auto = time.time()
df_timed_auto = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(CUSTOMERS_CSV)
)
count_auto = df_timed_auto.count()
time_auto  = time.time() - start_auto
print(f"inferSchema   : {count_auto} rows in {time_auto:.2f}s")

In [0]:
start_manual = time.time()
df_timed_manual = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(customers_schema)
    .load(CUSTOMERS_CSV)
)
count_manual = df_timed_manual.count()
time_manual  = time.time() - start_manual
print(f"Manual schema : {count_manual} rows in {time_manual:.2f}s")

In [0]:
# Comparison table — no changes needed
speedup = (time_auto - time_manual) / time_auto * 100 if time_auto > time_manual else 0.0
display(spark.createDataFrame([
    ("inferSchema=true",  f"{time_auto:.2f}s",    str(count_auto),   "2 scans (type detection + load)"),
    ("Manual StructType", f"{time_manual:.2f}s",  str(count_manual), "1 scan (schema known upfront)"),
], ["Method", "Time", "Rows", "Note"]))
print(f"Manual schema is faster by {speedup:.1f}% (on small data this may vary due to JVM warmup)")

In [0]:
# -- Validation --
assert count_auto == count_manual, "Both methods must return the same row count"
assert time_auto > 0 and time_manual > 0, "Timing values must be positive"
print(f"Task 5 OK: Both methods loaded {count_auto} rows. Timing recorded.")

## Task 6: read_files() — SQL-Native Unity Catalog Reader

Use `read_files()` — the SQL-native function for querying files directly from Unity Catalog Volumes. Include `_metadata.file_path` to track the file origin.

**What you need to do:** Write a `spark.sql()` query using `read_files()` to read customers. Alias `_metadata.file_path` as `file_path` in the output.

**Guidance — Task 06**

The goal is to read a file from a UC Volume using the SQL-native `read_files()` table function and access file metadata.

**read_files() syntax**
`read_files()` is a SQL table-valued function available in DBR 13.3+. It accepts a path and format options using SQL named argument syntax (note the `=>` arrow, not `=`):

```sql
SELECT *
FROM read_files('/Volumes/catalog/schema/volume/file.csv',
                format => 'csv',
                header => true)
```

**_metadata column**
Every file-based source in Spark exposes the `_metadata` struct column automatically. Useful fields:
- `_metadata.file_path` — full path to the source file
- `_metadata.file_name` — filename only
- `_metadata.file_size` — file size in bytes

Add `, _metadata.file_path AS file_path` to your SELECT to include the path in the result.

**read_files() vs spark.read**
`read_files()` is convenient for ad-hoc SQL queries and CTAS statements. `spark.read` is preferred when you need Python-level control (StructType schemas, complex transformations before writing). Both access the same data.

**Things to think about**
- Can `read_files()` be used inside a `CREATE TABLE ... AS SELECT` statement?
- What is the difference between `read_files()` and creating an external table over the same path?

In [0]:
customers_rf_df = spark.sql(f"""
    SELECT *, _metadata.file_path AS file_path
    FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
""")

display(customers_rf_df.limit(5))

In [0]:
# -- Validation --
assert customers_rf_df.count() > 0, "read_files() should return rows"
assert "file_path" in customers_rf_df.columns, "Expected 'file_path' column from _metadata.file_path"
print(f"Task 6 OK: {customers_rf_df.count()} rows via read_files(). Columns: {list(customers_rf_df.columns)}")

## Task 7: CTAS — Create Bronze Orders Table

Use CTAS (CREATE TABLE AS SELECT) to create a managed Delta table from the raw JSON orders file.

**What you need to do:** Write a `CREATE OR REPLACE TABLE` statement that reads all order columns plus `_metadata.file_path AS _source_file` from `ORDERS_JSON` and writes them to `{BRONZE_SCHEMA}.orders_bronze`.

**Guidance — Task 07**

The goal is to create a managed Delta table from a file source in a single SQL statement.

**CTAS pattern**
CTAS creates a table from a SELECT result in one operation — no need to define the schema first or write data separately:

```sql
CREATE OR REPLACE TABLE catalog.schema.table_name AS
SELECT *
FROM read_files('path', format => 'json')
```

**Key properties of CTAS**
- Creates a **managed Delta table** — Databricks controls the storage location
- `CREATE OR REPLACE` drops the existing table (if any) and creates a fresh one atomically
- The table schema is derived automatically from the SELECT result
- CTAS is **not incremental** — every re-run replaces the entire table

**Adding a source file column**
Include `_metadata.file_path AS _source_file` in the SELECT to record the file origin of each row. This is useful for debugging data quality issues later.

**Things to think about**
- If you run this CTAS statement twice, what happens to the data written in the first run?
- How is CTAS different from `INSERT INTO ... SELECT`?
- Is CTAS suitable for a daily scheduled load where you only want new files? Why?

In [0]:
ORDERS_BRONZE = f"{BRONZE_SCHEMA}.orders_bronze"

spark.sql(f"""
    CREATE OR REPLACE TABLE {ORDERS_BRONZE} AS
    SELECT *, _metadata.file_path AS _source_file
    FROM read_files('{ORDERS_JSON}', format => 'json')
""")

display(spark.table(ORDERS_BRONZE).limit(5))

In [0]:
# -- Validation --
orders_bronze_count = spark.table(ORDERS_BRONZE).count()
orders_bronze_cols  = spark.table(ORDERS_BRONZE).columns
assert orders_bronze_count > 0, "orders_bronze should not be empty"
assert "_source_file" in orders_bronze_cols, "Missing '_source_file' column"
print(f"Task 7 OK: orders_bronze created with {orders_bronze_count} rows. Columns: {list(orders_bronze_cols)}")

## Task 8: CTAS with Data Transformations

Create a `customers_bronze` table from the customers CSV. Apply transformations during load to normalize and enrich the data.

**What you need to do:** Write a CTAS with these transformations:
- `UPPER(first_name)`, `UPPER(last_name)` — normalize name fields to uppercase
- `LOWER(email)` — normalize email to lowercase
- `TO_DATE(registration_date, 'yyyy-MM-dd')` — cast string to Date type
- `DATEDIFF(current_date(), registration_date)` AS `days_since_registration`
- `current_timestamp()` AS `_ingestion_timestamp`
- Filter: `WHERE country IS NOT NULL`

**Guidance — Task 08**

The goal is to apply lightweight data transformations inside a CTAS statement to produce a clean Bronze table.

**Transformations inside CTAS**
A CTAS SELECT can contain any SQL expression: scalar functions, type casts, computed columns, and WHERE filters. The resulting column list becomes the table schema automatically — no DDL needed.

**Useful SQL functions for this task**
- `UPPER(col)` / `LOWER(col)` — text case normalization
- `TO_DATE(col, 'format')` — string to Date: `TO_DATE(registration_date, 'yyyy-MM-dd')`
- `DATEDIFF(end_date, start_date)` — integer number of days between two dates
- `current_timestamp()` — current UTC timestamp, no arguments needed

**Bronze vs Silver transformations**
It is a design choice whether to transform in Bronze or Silver. The common guideline: correct obvious format issues (case normalization, basic type casting) in Bronze; apply business logic and complex derivations in Silver.

**Things to think about**
- Customers with `country IS NULL` are filtered out — are they permanently lost?
- Why is it useful to add `_ingestion_timestamp` at the Bronze layer?
- If the `registration_date` format changes in the source file, what breaks?

In [0]:
CUSTOMERS_BRONZE = f"{BRONZE_SCHEMA}.customers_bronze"

spark.sql(f"""
    CREATE OR REPLACE TABLE {CUSTOMERS_BRONZE} AS
    SELECT
        customer_id,
        UPPER(first_name)                                      AS first_name,
        UPPER(last_name)                                       AS last_name,
        city,
        LOWER(email)                                           AS email,
        country,
        TO_DATE(registration_date, 'yyyy-MM-dd')               AS registration_date,
        DATEDIFF(current_date(), registration_date)            AS days_since_registration,
        current_timestamp()                                    AS _ingestion_timestamp
    FROM read_files('{CUSTOMERS_CSV}', format => 'csv', header => true)
    WHERE country IS NOT NULL
""")

display(spark.table(CUSTOMERS_BRONZE).limit(5))

In [0]:
# -- Validation --
cust_cols  = spark.table(CUSTOMERS_BRONZE).columns
cust_count = spark.table(CUSTOMERS_BRONZE).count()
assert "_ingestion_timestamp" in cust_cols, "Missing _ingestion_timestamp"
assert "days_since_registration" in cust_cols, "Missing days_since_registration"
sample_email = spark.table(CUSTOMERS_BRONZE).filter("email IS NOT NULL").select("email").first()[0]
assert sample_email == sample_email.lower(), f"Email should be lowercase, got: {sample_email}"
print(f"Task 8 OK: customers_bronze created — {cust_count} rows, transformations applied correctly")

## Summary

| Task | Topic | Key Point |
|------|-------|-----------|
| 1 | CSV inferSchema | Automatic types — two filesystem scans |
| 2 | CSV manual schema | StructType — one scan, guaranteed types |
| 3 | JSON partial schema | Read only needed columns; keep dates as StringType in Bronze |
| 4 | Parquet | Self-describing format — schema in file footer, no inference |
| 5 | Performance | inferSchema costs 2x I/O — use manual schema in production |
| 6 | read_files() | SQL-native UC Volume reader, _metadata columns available |
| 7 | CTAS basic | Creates managed Delta table from SELECT; not incremental |
| 8 | CTAS + transforms | Type casts, computed columns, filtering at ingest time |

> For **incremental ingestion** (COPY INTO, Auto Loader, Structured Streaming) — see **M05: Incremental Ingestion** (Day 2)

## Cleanup

In [0]:
for t in [
    f"{CATALOG}.{BRONZE_SCHEMA}.orders_bronze",
    f"{CATALOG}.{BRONZE_SCHEMA}.customers_bronze",
]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")
print("Lab cleanup complete")

## BONUS — PART 2 ADVANCED (If Time Permits)

> For participants with previous Spark/Databricks experience.
> Write your solutions from scratch — no scaffold code provided.
> Complete at least one of the two challenges.

### Challenge 1: Bug Hunt — Wrong Schema

The following code runs without errors but produces incorrect results. Find the bug, fix it, and explain what went wrong.

```python
bad_df = (spark.read
    .format("json")
    .option("inferSchema", "false")
    .load(ORDERS_JSON))

# This assertion fails:
assert dict(bad_df.dtypes)["total_amount"] == "double"
```

**Acceptance criteria:**
- Identify why `total_amount` is StringType instead of DoubleType
- Fix the read call with the correct approach
- Explain: what does `inferSchema=false` mean for JSON — is it the same behavior as for CSV?

In [0]:
# YOUR SOLUTION — Bug Hunt
# ------------------------------------------------------------

### Challenge 2: Design — Multi-Format Bronze Ingestion

Implement a function `ingest_to_bronze(dataset_path, catalog, schema)` that:
- Reads customers.csv, orders_batch.json, and products.parquet with explicit schemas
- Creates three Bronze tables: `customers_bronze2`, `orders_bronze2`, `products_bronze2`
- Adds `_source_file` and `_ingestion_timestamp` to every table
- Returns a dict with table names and row counts
- Is idempotent — safe to call multiple times (use `CREATE OR REPLACE`)

**Acceptance criteria:**
- All three tables created and non-empty
- `_source_file` and `_ingestion_timestamp` present in all tables
- Function documented with a short docstring

In [0]:
# YOUR SOLUTION — Multi-Format Bronze Ingestion
# ------------------------------------------------------------

← [02 — Data Reading & File Formats](../demo/02_data_ingestion.ipynb) | **[README](../../../README.md)** | [03 — Delta Lake →](../demo/03_delta_lake.ipynb)